In [ ]:
# Rooftop Geometry Analysis

In [3]:
# Load Data

import geopandas as gpd
import rasterio
from rasterio.sample import sample_gen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Load buildings
buildings = gpd.read_file("accra_buildings_combined.gpkg")
print(f"Loaded {len(buildings):,} buildings")

# Load rasters
dem_path = r'I:\GEO DATA ANALYSIS\solar_ghana\greater_accra_dem.tif'
slope_path = "accra_slope.tif"
aspect_path = "accra_aspect.tif"

print("Data loaded successfully.")

Loaded 632,195 buildings
Data loaded successfully.


In [6]:
# Extract Slope & Aspect per Building (Zonal Statistics)

import geopandas as gpd
import rasterio
import numpy as np
from tqdm import tqdm




UTM_CRS = "EPSG:32630"   
WGS84 = "EPSG:4326"


# Load buildings
print("Loading buildings...")
buildings = gpd.read_file("accra_buildings_combined.gpkg")

print(f"Original CRS: {buildings.crs}")


# Reproject to UTM for accuracy
print("Reprojecting buildings to UTM Zone 30N...")
buildings_utm = buildings.to_crs(UTM_CRS)
print(f"Now in CRS: {buildings_utm.crs}")


# Safe Centroid Sampling
def get_raster_value_at_centroid(gdf_utm, raster_path):
    """
    Extract raster values at building centroids
    (centroids computed in UTM → then converted to raster CRS)
    """
    values = []
    
    with rasterio.open(raster_path) as src:
        
        # Compute centroids in UTM 
        centroids_utm = gdf_utm.geometry.centroid
        
        # Convert to raster CRS
        centroids = gpd.GeoSeries(
            centroids_utm, crs=gdf_utm.crs
        ).to_crs(src.crs)
        
        # Sample raster
        for geom in tqdm(centroids, desc=f"Sampling {raster_path}"):
            x, y = geom.x, geom.y
            
            val = list(src.sample([(x, y)]))[0][0]
            
            if val is None or val == src.nodata:
                values.append(np.nan)
            else:
                values.append(val)
    
    return np.array(values)


# Extract Slope and Aspect
print("\nExtracting slope and aspect at building centroids...")

buildings['slope'] = get_raster_value_at_centroid(buildings_utm, slope_path)
buildings['aspect'] = get_raster_value_at_centroid(buildings_utm, aspect_path)


# Clean data
print("\nCleaning extracted values...")

# Clip unrealistic slope values
buildings['slope'] = buildings['slope'].clip(lower=0, upper=60)

# Fill missing aspect (flat roofs default south-facing)
buildings['aspect'] = buildings['aspect'].fillna(180)


# Summary stats
print("\nSlope & Aspect extraction completed!")
print(buildings[['slope', 'aspect']].describe())


# Save
output_path = "accra_buildings_with_slope_aspect.gpkg"
buildings.to_file(output_path, driver="GPKG")

print(f"\nSaved file: {output_path}")

Loading buildings...
Original CRS: EPSG:4326
Reprojecting buildings to UTM Zone 30N...
Now in CRS: EPSG:32630

Extracting slope and aspect at building centroids...


Sampling accra_aspect.tif: 100%|█████████████████████████████████████████████████████| 632195/632195 [02:20<00:00, 4485.89it/s]



Cleaning extracted values...

Slope & Aspect extraction completed!
               slope         aspect
count  630946.000000  632195.000000
mean       47.079010     158.519394
std        14.641202     102.320755
min         0.000000       0.000000
25%        35.264389      71.565048
50%        48.189686     153.434952
75%        60.000000     243.434952
max        60.000000     356.423676

Saved file: accra_buildings_with_slope_aspect.gpkg


In [21]:
# Calculate Usable Roof Area & Suitability

import numpy as np

print("\nCalculating geometry + weighted suitability score...")

# Reproject and area calculation
buildings_utm = buildings.to_crs("EPSG:32630")
buildings['footprint_area_m2'] = buildings_utm.geometry.area

# Roof utilization factor
buildings['roof_utilization_factor'] = np.where(
    buildings['slope'] <= 15, 0.92,
    np.where(buildings['slope'] <= 35, 0.75,
             np.clip(1 - (buildings['slope'] - 35) / 55, 0.35, 0.75))
)

buildings['usable_area_m2'] = (
    buildings['footprint_area_m2'] * 0.82 * buildings['roof_utilization_factor']
)

# Precompute Component Scores
print("Precomputing slope, aspect, and area scores...")

# 1. Slope Score
buildings['slope_score'] = np.where(
    buildings['slope'] <= 15, 100.0,
    np.where(buildings['slope'] <= 25, 80.0,
             np.where(buildings['slope'] <= 35, 50.0,
                      np.maximum(10.0, 100 - (buildings['slope'] - 35) * 3)))
)

# 2. Aspect Score
aspect_diff = np.minimum(np.abs(buildings['aspect'] - 180), 360 - np.abs(buildings['aspect'] - 180))
buildings['aspect_score'] = np.maximum(40.0, 100 - aspect_diff * 0.28)

# 3. Area Score (nonlinear)
buildings['area_score'] = 100 * (1 - np.exp(-buildings['usable_area_m2'] / 100))

# Main Weighted Score
w_aspect = 0.30
w_area = 0.25

buildings['suitability_score'] = round(
    w_slope * buildings['slope_score'] +
    w_aspect * buildings['aspect_score'] +
    w_area * buildings['area_score'], 1
)

print("\nMain Weighted Suitability Statistics:")
print(buildings['suitability_score'].describe().round(2))
print(f"Mean roof utilization factor: {buildings['roof_utilization_factor'].mean():.3f}")
print(f"Buildings with suitability > 70: {(buildings['suitability_score'] > 70).sum():,}")
print(f"Buildings with suitability > 80: {(buildings['suitability_score'] > 80).sum():,}")

# Sensitivity Analysis
print("\n---Sensitivity Analysis ---")

weights_list = [
    (0.50, 0.30, 0.20, "Heavy on Slope"),
    (0.45, 0.30, 0.25, "Balanced (Current)"),
    (0.40, 0.35, 0.25, "More on Aspect"),
    (0.50, 0.25, 0.25, "Strong Slope Focus")
]

for w_s, w_a, w_ar, name in weights_list:
    score_name = f"suitability_w{w_s}_{w_a}_{w_ar}"
    buildings[score_name] = round(
        w_s * buildings['slope_score'] +
        w_a * buildings['aspect_score'] +
        w_ar * buildings['area_score'], 1
    )
    mean_score = buildings[score_name].mean()
    high70 = (buildings[score_name] > 70).sum()
    print(f"{name:25} → Mean: {mean_score:.2f} | >70: {high70:,}")

# Show correlation between different weight schemes
print("\nCorrelation between weight variants:")
print(buildings[['suitability_score', 'suitability_w0.5_0.3_0.2', 
                 'suitability_w0.4_0.35_0.25']].corr().round(3))






Calculating geometry + weighted suitability score...
Precomputing slope, aspect, and area scores...

Main Weighted Suitability Statistics:
count    630946.00
mean         53.29
std          13.87
min          26.40
25%          42.00
50%          51.40
75%          63.20
max          95.90
Name: suitability_score, dtype: float64
Mean roof utilization factor: 0.669
Buildings with suitability > 70: 86,330
Buildings with suitability > 80: 23,811

---Sensitivity Analysis ---
Heavy on Slope            → Mean: 54.23 | >70: 102,598
Balanced (Current)        → Mean: 53.29 | >70: 86,330
More on Aspect            → Mean: 54.51 | >70: 87,654
Strong Slope Focus        → Mean: 52.06 | >70: 88,037

Correlation between weight variants:
                            suitability_score  suitability_w0.5_0.3_0.2  \
suitability_score                       1.000                     0.993   
suitability_w0.5_0.3_0.2                0.993                     1.000   
suitability_w0.4_0.35_0.25              0.9

In [23]:
# Save

print("\nSaving optimized weighted suitability dataset...")

# Ensure building_id exists
if 'building_id' not in buildings.columns:
    print("Adding building_id column...")
    buildings['building_id'] = range(1, len(buildings) + 1)

# Main output file
output_filename = "accra_buildings_with_weighted_suitability.gpkg"
buildings.to_file(output_filename, driver="GPKG")

print(f"Successfully saved main file as:")
print(f"   - {output_filename}")

# Summary CSV
summary_cols = ['building_id', 'footprint_area_m2', 'usable_area_m2', 
                'roof_utilization_factor', 'slope', 'aspect', 
                'slope_score', 'aspect_score', 'area_score', 'suitability_score']

buildings[summary_cols].to_csv("accra_buildings_geometry_summary.csv", index=False)

print(f"   - accra_buildings_geometry_summary.csv")

# Final summary
print("\n=== Final Summary ===")
print(f"Total buildings                  : {len(buildings):,}")
print(f"Mean usable roof area            : {buildings['usable_area_m2'].mean():.1f} m²")
print(f"Mean suitability score           : {buildings['suitability_score'].mean():.1f}/100")
print(f"High suitability buildings (>70) : {(buildings['suitability_score'] > 70).sum():,}")
print(f"Very high suitability (>80)      : {(buildings['suitability_score'] > 80).sum():,}")
print(f"Mean roof utilization factor     : {buildings['roof_utilization_factor'].mean():.3f}")


Saving optimized weighted suitability dataset...
Successfully saved main file as:
   - accra_buildings_with_weighted_suitability.gpkg
   - accra_buildings_geometry_summary.csv

=== Final Summary ===
Total buildings                  : 632,195
Mean usable roof area            : 53.7 m²
Mean suitability score           : 53.3/100
High suitability buildings (>70) : 86,330
Very high suitability (>80)      : 23,811
Mean roof utilization factor     : 0.669
